<a href="https://colab.research.google.com/github/bothwellishumba/Googlesheets/blob/main/Solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**A complete, step-by-step process on solving the case efficiently in  Google Sheets.**

### Part 1: Automated Summary Table from "Data" Sheet

1. **Go to your Google Sheet** that contains the "Data" sheet.

2. **Create a new sheet** for the summary (recommended name: `Summary`).

3. **In the `Summary` sheet**, set up the headers in row 1:
   - A1: `ID`
   - B1: `Driver`
   - C1: `Load Number`
   - D1: `Dispatcher`
   - E1: `Dispatcher Team`
   - F1: `Brokerage`
   - G1: `PU (Pickup)`
   - H1: `Stops`
   - I1: `DO (Drop-off)`
   - J1: `DO Date`
   - K1: `Reconciled`
   - L1: `Notes`
   - M1: `Created On`
   - N1: `Status`

4. **Pull the data dynamically using `FILTER` + `ARRAYFORMULA`** (best practice for scalability):

   In cell **A2** of the Summary sheet, paste this single formula:

   ```google-sheets
   =FILTER(
     Data!A:N,
     Data!A:A <> ""   // Adjust the column if your ID column is different
   )
   ```

   This will automatically pull **all rows** from the Data sheet (columns A to N) while skipping completely blank rows. As new data is added to the "Data" sheet, the summary will update instantly.

   **Alternative (more explicit column mapping if column order in Data differs):**
   ```google-sheets
   =QUERY(Data!A:Z,
     "SELECT Col1, Col2, Col3, Col4, Col5, Col6, Col7, Col8, Col9, Col10, Col11, Col12, Col13, Col14
      WHERE Col1 IS NOT NULL", 1)
   ```

   Adjust the `ColX` numbers to match the actual column positions of the required fields in the Data sheet.

5. **Formatting & Best Practices**:
   - Format column J (`DO Date`) and M (`Created On`) as **Date** format.
   - Freeze row 1 (`View > Freeze > 1 row`).
   - Apply alternating row colors for readability.
   - Optional: Add a timestamp in another cell showing last update: `=NOW()` (format as Date time).

This solution is fully dynamic — no manual copying required.

---

### Part 2: REPORT Sheet with Issue Duration & Ranking

#### Step A: Create Helper Calculations (Recommended in Data sheet or a hidden Helper sheet)

Add these calculated columns at the end of the **Data** sheet (or in a new Helper sheet that references Data).

Assume:
- Column for **Load Issue** (e.g., column O or wherever it is).
- Column for **Delivery Date** (e.g., column P).

**Add these formulas (starting from the first data row, e.g., row 2):**

- **Issue Duration (days)** — Column Q (example):

   ```google-sheets
   =ARRAYFORMULA(
     IF(ROW()=1, "Issue Duration",
       IF(
         Data!O2:O = "BOL",
         MAX(0, DATE(2025,2,13) - Data!P2:P),   // BOL: all days

         IF(
           Data!O2:O = "Awaiting Money",
           IF(DATE(2025,2,13) - Data!P2:P >= 10,
              DATE(2025,2,13) - Data!P2:P,
              ""),                          // Only >=10 days
           ""
         )
       )
     )
   )
   ```

This creates a clean "Issue Duration" column that follows the exact business rules.

---

#### Step B: Create the “REPORT” Sheet

1. Insert a new sheet and name it **`REPORT`**.

2. Build the ranking table. Use `QUERY` for a clean, dynamic summary.

**Recommended layout in REPORT sheet:**

| Team              | Dispatcher       | Loads with Issues | Total Duration (days) |
|-------------------|------------------|-------------------|-----------------------|

**Formula for the full report** (paste in A1 of REPORT sheet):

```google-sheets
=QUERY(
  {
    Data!E:E,                // Dispatcher Team
    Data!D:D,                // Dispatcher
    IF(Data!O:O <> "", 1, 0), // Flag loads with issues
    IFERROR(Q:Q, 0)          // Issue Duration column you created
  },
  "SELECT Col1, Col2, COUNT(Col3), SUM(Col4)
   WHERE Col3 = 1 AND Col4 > 0
   GROUP BY Col1, Col2
   ORDER BY SUM(Col4) DESC, COUNT(Col3) DESC
   LABEL Col1 'Team', Col2 'Dispatcher', COUNT(Col3) 'Loads with Issues', SUM(Col4) 'Total Duration (days)'",
  1
)
```

**Key points of this QUERY**:
- Groups by **Team** and **Dispatcher**.
- Counts only loads that have issues.
- Sums only the qualifying durations.
- Ranks by **Total Duration** descending, then by count.
- Automatically updates when new data arrives.

---

### Final Recommendations for Robustness

- **Protect** the Summary and REPORT sheets (right-click sheet tab → Protect sheet) so formulas aren't accidentally broken.
- Use **named ranges** for key columns (Data!Delivery_Date, Data!Load_Issue, etc.) to make formulas more readable.
- Add a dashboard-style title and date: `="Issue Report as of "&TEXT(TODAY(),"mmmm dd, yyyy")`.
- Consider using **Filter views** or **Slicers** on the Summary sheet for quick analysis.
- If data volume becomes very large (>50k rows), consider moving to Google Data Studio / Looker Studio later, but this formula approach scales well into thousands of rows.

This solution is **100% formula-based**, fully automated, and follows Google Sheets best practices for maintainability.

